# Nemotron-3-Nano-30B LoRA — Submission Demo

Loads a pre-trained LoRA adapter (trained off-Kaggle on a GB10 machine) and saves it to
`/kaggle/working` so the output can be submitted to the competition.

**This notebook does not train.** Training was done off-Kaggle using `scripts/train_lora.py`
inside a Docker container based on `Dockerfile.gb10-26-01`. See the
[prize eligibility notebook](https://www.kaggle.com/code/gdataranger/nemotron-3-nano-30b-lora-reasoning-challenge)
for full methodology.


## 1. Environment setup


In [ ]:
import site

# CUTLASS DSL — required for Nemotron-H Mamba-2 kernels on Kaggle
cutlass_pkg_path = (
    "/kaggle/usr/lib/notebooks/ryanholbrook/"
    "nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
)
site.addsitedir(cutlass_pkg_path)

# transformers >= 5.3.0 has native NemotronH support — no trust_remote_code or mamba_ssm needed
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--no-warn-conflicts", "transformers==5.5.3"], check=True)

import kagglehub
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Load base model

The base model is loaded from Kaggle's model hub (no HF token needed inside Kaggle).


In [ ]:
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print("Base model path:", MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
base_model.config.use_cache = False
print("Base model loaded.")


## 3. Load pre-trained LoRA adapter

The adapter was trained off-Kaggle. Set the source below:
- **Option A** — Hugging Face Hub: set `ADAPTER_SOURCE = "hf"` and fill `HF_ADAPTER_REPO`
- **Option B** — Kaggle dataset: upload adapter files as a dataset, attach it to this notebook,
  set `ADAPTER_SOURCE = "kaggle"` and fill `KAGGLE_ADAPTER_PATH`


In [ ]:
# --- Configure adapter source ---
ADAPTER_SOURCE = "hf"  # "hf" or "kaggle"

HF_ADAPTER_REPO    = "marksusol/nemotron-nano-30b-lora-reasoning"  # TODO: publish
KAGGLE_ADAPTER_PATH = "/kaggle/input/nemotron-lora-adapter"          # if ADAPTER_SOURCE="kaggle"

OUTPUT_DIR = "/kaggle/working"

# --- Load adapter ---
if ADAPTER_SOURCE == "hf":
    adapter_path = HF_ADAPTER_REPO
    print(f"Loading adapter from HF Hub: {adapter_path}")
elif ADAPTER_SOURCE == "kaggle":
    adapter_path = KAGGLE_ADAPTER_PATH
    print(f"Loading adapter from Kaggle dataset: {adapter_path}")
else:
    raise ValueError(f"Unknown ADAPTER_SOURCE: {ADAPTER_SOURCE}")

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("Adapter loaded.")
model.print_trainable_parameters()


## 4. Smoke test — verify \\boxed{} format


In [ ]:
SYSTEM_PROMPT = (
    "You are a careful reasoning model. "
    "Solve the problem step by step and end with Final answer: \\boxed{...}."
)

def generate(problem: str, max_new_tokens: int = 256) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": problem},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = "\n".join(f"{m['role']}: {m['content']}" for m in messages) + "\nassistant:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=1.0)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

answer = generate("What is 17 + 25?")
print(answer)
assert "boxed" in answer, "WARNING: response missing \\boxed{} — check adapter load"


## 5. Save adapter to /kaggle/working

Saving to `/kaggle/working` makes the adapter files available as notebook output.
After this cell runs, go to **Output** → download or submit the files to the competition.


In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved adapter to {OUTPUT_DIR}")

for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}  ({size:,} bytes)")
